Imports and configuration

In [ ]:
import kagglehub
import re
import os
import pandas as pd
import tiktoken
from datasets import load_dataset


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# configuration for a GPT-2-style model with approximately 124 million parameters
GPT_CONFIG_124M = {
    "vocab_size": 50257, # vocabulary size
    "context_length": 1024, # context length
    "emb_dim": 768, # embedding dimension
    "n_heads": 12, # number of attention heads
    "n_layers": 12, # number of layers
    "drop_rate": 0.1, # dropout rate
    "qkv_bias": False # query-key-value bias
}

Dataset class

In [ ]:
class GPTDataset(Dataset):
  '''
  Dataset class for preparing input-target token sequences.
  '''
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # convert the corpus into token IDs
    token_ids = tokenizer.encode(txt)

    # create fixed-length sequences using a sliding window
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]

      # next-token prediction
      target_chunk = token_ids[i + 1:i + max_length + 1]

      # store sequences as PyTorch tensors
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    # return the number of input-target pair sequence pairs
    return len(self.input_ids)

  def __getitem__(self, idx):
    # receive an input sequence and its corresponding target sequence
    return self.input_ids[idx], self.target_ids[idx]



Data Preparation Class

In [3]:
class DataPreparation:
    '''
    Handles dataset loading, preprocessing, splitting, and batching.
    '''
    def __init__(self):
        # initialize the GPT-2 tokenizer
        self.tokenizer = tiktoken.get_encoding("gpt2")

    def load_dataset(self):
        # load the Hugging Face dataset
        ds = load_dataset("MoreThought/Fable-5.1-Max-Reasoning-Filtered-10000x")
        print("Dataset keys available:", ds["train"].column_names)
        print("Example row:", ds["train"][0])

        # use the 'train' split of the dataset
        train_split = ds["train"]

        # collect text from message content fields
        texts = []

        for row in train_split:
            if "messages" in row and isinstance(row["messages"], list):
                for message in row["messages"]:
                    content = message.get("content", "")

                    # ignore empty messages
                    if content:
                        texts.append(content)

        # combine messages into one continuous text corpus
        words = " ".join(texts)
        return words

    def split_dataset(self, words, train_ratio=0.90):
        # determine the split position based on character count
        split_idx = int(train_ratio * len(words))

        # allocate 90% to training and 10% to validation
        train_data = words[:split_idx]
        val_data = words[split_idx:]

        return train_data, val_data

    def create_dataloader(self, txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
        # convert text into input-target token sequences
        dataset = GPTDataset(txt, self.tokenizer, max_length, stride)

        # create batches for model training or evaluation
        dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            drop_last=drop_last,
            num_workers=num_workers
        )

        return dataloader

Multi-head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
  '''
  Implements the multi-head self attention mechanism.
  '''
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    # ensure the embedding dimension can be divided evenly across heads
    assert (d_out % num_heads == 0), \
    "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads

    # learnable linear projections for queries, keys, and values
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # combine information from all attention heads
    self.out_proj = nn.Linear(d_out, d_out)

    # apply dropout to attention weights
    self.dropout = nn.Dropout(dropout)

    # register an upper-triangular causal mask to prevent attention to future tokens
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    # extract batch size, sequence length, and input embedding dimensions
    b, num_tokens, d_in = x.shape

    # project input embeddings into kqv representations
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # split each projectin into multiple attention heads
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    # rearrange dimensions to process attention heads independently
    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    # compute attention scores using query-key dot products
    attn_scores = queries @ keys.transpose(2, 3)

    # select the mausal mask for the current sequence length
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    # prevent tokens from attending to future positions
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    # scale attention scores and normalize them into probabilities
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    # apply dropout to probabilities
    attn_weights = self.dropout(attn_weights)

    # compute weighted sums of value vectors for each attn head
    context_vec = (attn_weights @ values).transpose(1,2)

    # concatenate attn head outputs into the original output dimension
    context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

    # apply the final output projection
    context_vec = self.out_proj(context_vec)

    return context_vec, attn_weights


Layer Normalization

In [ ]:
class LayerNorm(nn.Module):
    '''
    Normalizes token representations across teh embedding dimension.
    '''
    def __init__(self, emb_dim):
        super().__init__()

        # prevent division by 0
        self.eps = 1e-5

        # learnable scale and shift parameters
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.bias = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # calculate mean and variance across the embedding dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # normalize activations to approximately zero mean and unit variance
        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        # apply learnable scaling and shifting
        return self.scale * norm_x + self.bias

GELU

In [ ]:
class GELU(nn.Module):
    '''
    Gaussian Error Linear Unit activation function.
    '''
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # apply the tanh approximation of GELU for nonlinear transformations
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

Feedforward Network

In [ ]:
class FeedForward(nn.Module):
    '''
    Implements the feedforward network for transformer blocks.
    '''
    def __init__(self, cfg):
        super().__init__()

        self.layers = nn.Sequential(
            # expand the embedding dimension by a factor of 4
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            # apply nonlinear activation
            GELU(),
            # project representations back tot he original embedding dimension
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
  '''
  Transformer block combining self-attention and a feedforward network.
  '''
  def __init__(self,cfg):
    super().__init__()

    # initialize causal multi-head self-attn
    self.att = MultiHeadAttention(
        d_in=cfg["emb_dim"],
        d_out=cfg["emb_dim"],
        context_length=cfg["context_length"],
        num_heads=cfg["n_heads"],
        dropout=cfg["drop_rate"],
        qkv_bias=cfg["qkv_bias"]
    )

    # initialize feedforward network
    self.ff = FeedForward(cfg)

    # separate normalization layers for attention and feedforward operations
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])

    # dropout applied before residual connections
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # preserve original input for the attn residual connection
    shortcut = x

    # normalize inputs
    x = self.norm1(x)

    # comput attn output; only take the context_vec, discard attn weights
    x, _ = self.att(x)

    # apply dropout and add the original input through a residual connection
    x = self.drop_shortcut(x)
    x = x + shortcut

    # preserve attn output for the feedforward residual connection
    shortcut = x

    # normalize and process representations through the network
    x = self.norm2(x)
    x = self.ff(x)

    # apply dropout and the second residual connection
    x = self.drop_shortcut(x)
    x = x + shortcut
    return x

GPT Architecture

In [ ]:
class GPTModel(nn.Module):
  '''
  Implements the complete decoder-only GPT language model.
  '''
  def __init__(self, cfg):
    super().__init__()

    # convert token IDs into dense vector representations
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

    # learn positional embeddings to represent token order
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

    # regularize combined token and positional embeddings
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # stack transformer blocks
    self.trf_blocks = nn.Sequential(
        *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

    # normalize final transformer representations
    self.final_norm = LayerNorm(cfg["emb_dim"])

    # project hidden representations into vocabulary-sized logits
    self.out_head = nn.Linear(
        cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape

    # look up token embeddings
    tok_embeds = self.tok_emb(in_idx)

    # generate positional embeddings for each token position
    pos_embeds = self.pos_emb(
        torch.arange(seq_len, device=in_idx.device)
        )
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)

    # pass embeddings through transformer layers
    x = self.trf_blocks(x)

    # normalize the final hidden states
    x = self.final_norm(x)

    # generate unnormalized vocabulary scores for every token position
    logits = self.out_head(x)
    return logits

Text Generation Class

In [ ]:
class TextGenerator:
    '''
    Handles token conversion and autoregressive text generation.
    '''
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def text_to_token_ids(self, text):
        # encode text into toekn IDs
        encoded = self.tokenizer.encode(text, allowed_special={'<|endoftext|>'})

        # add a batch dimension to the token sequence
        encoded_tensor = torch.tensor(encoded).unsqueeze(0)

        return encoded_tensor

    def token_ids_to_text(self, token_ids):
        # remove the batch dimension
        flat = token_ids.squeeze(0)

        # decode token IDs into readable text
        return self.tokenizer.decode(flat.tolist())

    def generate_text(self, model, idx, max_new_tokens, context_size):
        # generate one token at a time using previous tokens as context
        for _ in range(max_new_tokens):
            # restrict input to the model's maximum context window
            idx_cond = idx[:, -context_size:]

            # disable gradient tracking
            with torch.no_grad():
                logits = model(idx_cond)

            # select vocab logits for the final token position
            logits = logits[:, -1, :]

            # convert logits into a probability distribution
            probas = torch.softmax(logits, dim=-1)

            # select the tokens with the highest probability
            idx_next = torch.argmax(probas, dim=-1, keepdim=True)

            # append the predicted token to the existing sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

    def generate_and_print_sample(self, model, device, start_context):
        model.eval()

        # retrieve the maximum context length
        context_size = model.pos_emb.weight.shape[0]

        # tokenize the starting prompt and move it to the device
        encoded = self.text_to_token_ids(start_context).to(device)

        # generate 50 additional tokens without computing gradients
        with torch.no_grad():
            token_ids = self.generate_text(
                model=model,
                idx=encoded,
                max_new_tokens=50,
                context_size=context_size
            )

        # decode and print the generated sequence
        decoded_text = self.token_ids_to_text(token_ids)
        print(decoded_text.replace("\n", " "))

        # restore training mode
        model.train()

GPT Trainer Class

In [ ]:
class GPTTrainer:
    '''
    Manages model training, loss calculation, and evaluation.
    '''
    def __init__(self, model, train_loader, val_loader,
                 optimizer, device, tokenizer):

        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.device = device

        # initialize the text generator for inspecting model predictions
        self.generator = TextGenerator(tokenizer)

    def calc_loss_batch(self, input_batch, target_batch):
        input_batch = input_batch.to(self.device)
        target_batch = target_batch.to(self.device)

        # compute vocab logits for all input token positions
        logits = self.model(input_batch)

        # flatten batch and sequence dimensions
        # compare predicted token distributions against teh target token IDs
        loss = torch.nn.functional.cross_entropy(
            logits.flatten(0, 1),
            target_batch.flatten()
        )

        return loss

    def calc_loss_loader(self, data_loader, num_batches=None):
        total_loss = 0.

        # if the dataloader contains no batches
        if len(data_loader) == 0:
            return float("nan")

        # evaluate the entire dataloader if no batch limit is specified
        elif num_batches is None:
            num_batches = len(data_loader)
        else:

            # prevent evaluation from exceeding available batches
            num_batches = min(num_batches, len(data_loader))

        # calculate and accumulate loss for selected batch
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i < num_batches:
                loss = self.calc_loss_batch(
                    input_batch,
                    target_batch
                )
                total_loss += loss.item()
            else:
                break

        # return average cross-entropy loss across evaluated batches
        return total_loss / num_batches

    def evaluate_model(self, eval_iter):
        self.model.eval()

        # evaluate without storing gradients
        with torch.no_grad():

            # calculate training loss
            train_loss = self.calc_loss_loader(
                self.train_loader,
                num_batches=eval_iter
            )

            # caclulate validation loss
            val_loss = self.calc_loss_loader(
                self.val_loader,
                num_batches=eval_iter
            )
        self.model.train()

        return train_loss, val_loss

    def train_model(self, num_epochs, eval_freq,
                    eval_iter, start_context):

        train_losses, val_losses, track_tokens_seen = [], [], []

        # track the number of processed tokens and optimization steps
        tokens_seen, global_step = 0, -1

        for epoch in range(num_epochs):
            self.model.train()

            for input_batch, target_batch in self.train_loader:
                # clear gradients from the previous optimization step
                self.optimizer.zero_grad()

                # calculate next-token prediction loss
                loss = self.calc_loss_batch(
                    input_batch,
                    target_batch
                )

                # compute gradients through backpropagation
                loss.backward()

                # update model parameters using the optimizer
                self.optimizer.step()
                tokens_seen += input_batch.numel()
                global_step += 1

                # evaluate model performance at regular intervals
                if global_step % eval_freq == 0:
                    train_loss, val_loss = self.evaluate_model(
                        eval_iter
                    )

                    # record loss and cumulative tokens processed
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)

                    # print training progress
                    print(
                        f"Ep {epoch+1} (Step {global_step:06d}): "
                        f"Train loss {train_loss:.3f}, "
                        f"Val loss {val_loss:.3f}"
                    )

            # generate sample text after each epoch to inspect model behavior
            self.generator.generate_and_print_sample(
                self.model,
                self.device,
                start_context
            )

        # return training history
        return train_losses, val_losses, track_tokens_seen

Main Execution

In [ ]:
from google.colab import drive
import os

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# prepare dataset
data = DataPreparation()
words = data.load_dataset()
print(f"Total characters in loaded text: {len(words)}")

train_data, val_data = data.split_dataset(words)
print(f"Train characters: {len(train_data)}, Val characters: {len(val_data)}")

# ensure we have enough data
if len(words) < 5000:
    print("Warning: The extracted text dataset is extremely small or empty. Please check your dataset parser.")

# create training batches of 8 sequences, each containing 1024 tokens
train_loader = data.create_dataloader(
    train_data,
    batch_size=8,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

# create validatino batches
val_loader = data.create_dataloader(
    val_data,
    batch_size=8,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# random seed for reproducable model initialization
torch.manual_seed(123)

# model initialization
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
print("Model initialized.")

# optimizer intialization for updating model parameters
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0004,
    weight_decay=0.1
)

# initialize trainer with the model, datasets, optimizer, and tokenizer
trainer = GPTTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    tokenizer=data.tokenizer
)

# verify that training and validation dataloaders contain batches
if len(train_loader) > 0 and len(val_loader) > 0:
    # disable dropout and gradient tracking
    model.eval()
    with torch.no_grad():
        # estimate initial training and validation losses using 5 batches
        train_loss = trainer.calc_loss_loader(train_loader, num_batches=5)
        val_loss = trainer.calc_loss_loader(val_loader, num_batches=5)
    print("Training loss:", train_loss)
    print("Validation loss:", val_loss)

    # Train model for 1 epoch
    num_epochs = 1
    train_losses, val_losses, tokens_seen = trainer.train_model(
        num_epochs=num_epochs,
        eval_freq=5,
        eval_iter=5,
        start_context="The data seems to"
    )
else:
    # stop training if the dataset cannot produce complete batches
    print("Data loaders are empty. Training aborted. Please feed more text or decrease context_length.")

# mount Google Drive
drive.mount('/content/drive')

# define the destination directory on Google Drive
drive_dir = '/content/drive/MyDrive/GPT_Model_Checkpoints'
os.makedirs(drive_dir, exist_ok=True)

save_path = os.path.join(drive_dir, 'model_and_optimizer_.pth')

# save the checkpoint to Google Drive
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}, save_path)

print(f'Model checkpoint saved successfully to: {save_path}')

README.md:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

5k/train.jsonl: reconstructing file:   0%|          |  0.00B /  565MB            

5k/train.jsonl: downloading bytes:           |  0.00B            

10k/train.jsonl: reconstructing file:   0%|          |  0.00B / 1.97GB            

10k/train.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating full split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset keys available: ['messages', 'tools']
Example row: {'messages': [{'role': 'system', 'content': "You are an autonomous coding agent running on the user's machine.\n\nYou work in the current directory of a real repository. Use your tools to read, search, create, and edit files and to run shell commands (tests, builds, type-checkers). Some tools are deferred: only their names are known until you load their schemas with ToolSearch. Depending on the environment, some tools may be unavailable or without network access; if a tool call fails, adapt and continue with what is available. Never guess at the contents of a file you have not read, and never claim a fix works without running the verification yourself.\n\nMethod:\n1. Reproduce first. Read the relevant code and run the failing command before changing anything.\n2. Form a hypothesis about the root cause; make the smallest edit that tests it.\n3. Verify. Rerun the tests/build after every meaningful change. If it still fails, re-re

ValueError: mount failed